In [ ]:
# CELL 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
# CELL 2: Imports and constants
import os, cv2, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

DATA_ROOT  = '/content/avec2014/AVEC2014'
FRAME_ROOT = '/content/avec2014_frames'
QR_CKPT    = '/content/drive/MyDrive/quantile_checkpoint_epoch7.pth'
GENDER_CSV = '/content/drive/MyDrive/avec2014_gender.csv'
SAVE_DIR   = '/content/drive/MyDrive/avec2014_mondrian_recal'
os.makedirs(SAVE_DIR, exist_ok=True)

CLIP_LEN    = 16
STRIDE      = 32
N_QUANTILES = 99
M_BINS      = 4
ALPHA       = 0.1
TARGET      = 1 - ALPHA
BATCH_CLIPS = 16   # how many clips to batch per forward pass on CPU

marker = os.path.join(DATA_ROOT, 'labels.csv')
if not os.path.exists(marker):
    print('Extracting dataset...')
    with zipfile.ZipFile('/content/drive/MyDrive/AVEC2014.zip', 'r') as z:
        z.extractall('/content/avec2014')
    print('Done.')
else:
    print('Dataset already extracted.')

print(f'QR_CKPT exists: {os.path.exists(QR_CKPT)}')

Device: cpu
Dataset already extracted.
QR_CKPT exists: True


In [ ]:
# CELL 3: Load labels, gender map, collect videos
def load_labels():
    df = pd.read_csv(os.path.join(DATA_ROOT, 'labels.csv'))
    labels = {}
    for _, row in df.iterrows():
        key = str(row['filename']).strip().replace('\\', '/')
        key = os.path.splitext(key)[0]
        labels[key] = float(row['BDI-II'])
    return labels

def load_gender_map():
    df = pd.read_csv(GENDER_CSV)
    gmap = {}
    for _, row in df.iterrows():
        key = str(row['filename']).strip().replace('\\', '/')
        key = os.path.splitext(key)[0]
        gmap[key] = str(row['gender']).strip().upper()
    return gmap

def collect_videos(folders, labels):
    if isinstance(folders, str):
        folders = [folders]
    items = []
    for folder in folders:
        if not os.path.exists(folder):
            continue
        for root, _, files in os.walk(folder):
            for f in sorted(files):
                if not f.lower().endswith('.mp4'):
                    continue
                path = os.path.join(root, f)
                rel  = os.path.relpath(path, DATA_ROOT).replace('\\', '/')
                stem = os.path.splitext(rel)[0]
                if stem in labels:
                    items.append((path, stem, labels[stem]))
    return items

labels      = load_labels()
gender_map  = load_gender_map()

TRAIN_DIRS = [os.path.join(DATA_ROOT, 'Training'), os.path.join(DATA_ROOT, 'Development')]
CAL_DIR    = os.path.join(DATA_ROOT, 'Testing', 'Northwind')
TEST_DIR   = os.path.join(DATA_ROOT, 'Testing', 'Freeform')

train_items = collect_videos(TRAIN_DIRS, labels)
cal_items   = collect_videos(CAL_DIR,    labels)
test_items  = collect_videos(TEST_DIR,   labels)

print(f'Train: {len(train_items)}  Cal: {len(cal_items)}  Test: {len(test_items)}')

train_labels_arr = np.array([l for _, _, l in train_items])
LABEL_MEAN = float(train_labels_arr.mean())
LABEL_STD  = float(train_labels_arr.std())
print(f'Label norm — mean: {LABEL_MEAN:.2f}  std: {LABEL_STD:.2f}')

Train: 200  Cal: 50  Test: 50
Label norm — mean: 15.34  std: 12.07


In [ ]:
# CELL 4: Extract frames (skips if already done)
os.makedirs(FRAME_ROOT, exist_ok=True)

def extract_video(path, stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) >= CLIP_LEN:
        return
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(path)
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (112, 112))
        cv2.imwrite(os.path.join(out_dir, f'{idx:05d}.jpg'), frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        idx += 1
    cap.release()

# Only cal + test needed for this notebook (no training)
needed_items = cal_items + test_items
print(f'Extracting frames for {len(needed_items)} videos (cal + test only)...')
for path, stem, label in tqdm(needed_items):
    extract_video(path, stem)
print('Extraction complete.')

Extracting frames for 100 videos (cal + test only)...


100%|██████████| 100/100 [00:00<00:00, 681.57it/s]

Extraction complete.


In [ ]:
# CELL 5: Dataset — returns list of clip tensors per video (not pre-stacked)
def load_frames_from_ssd(stem, start, n=CLIP_LEN):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    frames  = []
    for i in range(start, start + n):
        fpath = os.path.join(out_dir, f'{i:05d}.jpg')
        if not os.path.exists(fpath):
            break
        frame = cv2.imread(fpath)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    return frames

def count_frames(stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if not os.path.exists(out_dir):
        return 0
    return len([f for f in os.listdir(out_dir) if f.endswith('.jpg')])

def frames_to_tensor(frames):
    arr = np.stack(frames, axis=0).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr.transpose(3, 0, 1, 2)
    return torch.from_numpy(arr)

def get_clip_starts(T, stride=STRIDE):
    return list(range(0, T - CLIP_LEN + 1, stride))

class AVEC2014Eval(Dataset):
    def __init__(self, items):
        self.samples = []
        for path, stem, label in items:
            T = count_frames(stem)
            if T >= CLIP_LEN:
                self.samples.append((stem, label))
        print(f'Eval videos: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        stem, label = self.samples[idx]
        T     = count_frames(stem)
        clips = []
        for s in get_clip_starts(T, stride=8):
            frames = load_frames_from_ssd(stem, s, CLIP_LEN)
            if len(frames) < CLIP_LEN:
                continue
            clips.append(frames_to_tensor(frames))
        if not clips:
            clips.append(torch.zeros(3, CLIP_LEN, 112, 112))
        return torch.stack(clips), torch.tensor(label, dtype=torch.float32), stem

cal_dataset  = AVEC2014Eval(cal_items)
test_dataset = AVEC2014Eval(test_items)

Eval videos: 50
Eval videos: 50


In [ ]:
# CELL 6: C3D Quantile model and load checkpoint
class C3DQuantile(nn.Module):
    def __init__(self, n_quantiles=99, dropout=0.5):
        super().__init__()
        self.conv1  = nn.Conv3d(3,   64,  kernel_size=(3,3,3), padding=(1,1,1))
        self.pool1  = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.conv2  = nn.Conv3d(64,  128, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool2  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool3  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool4  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool5  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2), padding=(0,1,1))
        self.relu    = nn.ReLU(inplace=False)
        self.fc6     = nn.Linear(8192, 4096)
        self.fc7     = nn.Linear(4096, 64)
        self.fc8     = nn.Linear(64, n_quantiles)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.relu(self.conv1(x));  x = self.pool1(x)
        x = self.relu(self.conv2(x));  x = self.pool2(x)
        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x)); x = self.pool3(x)
        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x)); x = self.pool4(x)
        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x)); x = self.pool5(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc6(x)))
        x = self.dropout(self.relu(self.fc7(x)))
        return self.fc8(x)

qr_model = C3DQuantile(n_quantiles=N_QUANTILES).to(device)
qr_model.load_state_dict(torch.load(QR_CKPT, map_location=device))
qr_model.eval()
torch.set_num_threads(os.cpu_count())
print(f'Quantile model loaded. CPU threads: {os.cpu_count()}')

Quantile model loaded. CPU threads: 2


In [ ]:
# CELL 7 replacement — smaller batches, finer progress logging
def get_video_quantiles_fast(dataset, model, batch_clips=8):
    model.eval()
    results = []
    with torch.no_grad():
        for idx in range(len(dataset)):
            clips, label, stem = dataset[idx]
            print(f'  Video {idx+1}/{len(dataset)}: {stem} — {len(clips)} clips', flush=True)
            all_preds = []
            for i in range(0, len(clips), batch_clips):
                chunk = clips[i:i+batch_clips].to(device)
                preds = model(chunk)
                all_preds.append(preds)
            q_norm = torch.cat(all_preds).mean(dim=0).numpy()
            q_pred = q_norm * LABEL_STD + LABEL_MEAN
            results.append((stem, label.item(), q_pred))
    return results

print('Evaluating calibration set...')
cal_results = get_video_quantiles_fast(cal_dataset, qr_model, batch_clips=8)
print(f'Cal done: {len(cal_results)} videos')

print('\nEvaluating test set...')
test_results = get_video_quantiles_fast(test_dataset, qr_model, batch_clips=8)
print(f'Test done: {len(test_results)} videos')

Evaluating calibration set...
  Video 1/50: Testing/Northwind/203_2_Northwind_video — 119 clips
  Video 2/50: Testing/Northwind/206_2_Northwind_video — 126 clips
  Video 3/50: Testing/Northwind/210_2_Northwind_video — 160 clips
  Video 4/50: Testing/Northwind/211_2_Northwind_video — 171 clips
  Video 5/50: Testing/Northwind/212_1_Northwind_video — 122 clips
  Video 6/50: Testing/Northwind/214_3_Northwind_video — 160 clips
  Video 7/50: Testing/Northwind/218_3_Northwind_video — 145 clips
  Video 8/50: Testing/Northwind/220_1_Northwind_video — 149 clips
  Video 9/50: Testing/Northwind/220_3_Northwind_video — 145 clips
  Video 10/50: Testing/Northwind/224_1_Northwind_video — 242 clips
  Video 11/50: Testing/Northwind/226_2_Northwind_video — 152 clips
  Video 12/50: Testing/Northwind/234_2_Northwind_video — 160 clips
  Video 13/50: Testing/Northwind/236_3_Northwind_video — 152 clips
  Video 14/50: Testing/Northwind/237_1_Northwind_video — 175 clips
  Video 15/50: Testing/Northwind/240_3_No

In [ ]:
# CELL 8: Save raw quantile arrays immediately (safety checkpoint)
# So if anything crashes after this point, we don't need to re-evaluate.

import pickle
with open(os.path.join(SAVE_DIR, 'cal_results.pkl'), 'wb') as f:
    pickle.dump(cal_results, f)
with open(os.path.join(SAVE_DIR, 'test_results.pkl'), 'wb') as f:
    pickle.dump(test_results, f)
print('Raw quantile results saved to Drive (safety checkpoint).')
print('If session disconnects from here on, reload with:')
print("  cal_results = pickle.load(open('.../cal_results.pkl','rb'))")
print("  test_results = pickle.load(open('.../test_results.pkl','rb'))")

Raw quantile results saved to Drive (safety checkpoint).
If session disconnects from here on, reload with:
  cal_results = pickle.load(open('.../cal_results.pkl','rb'))
  test_results = pickle.load(open('.../test_results.pkl','rb'))


In [ ]:
# CELL 9: Global CQR baseline + FUQ(gender) baseline for comparison
QUANTILES = torch.linspace(0.01, 0.99, N_QUANTILES).to(device)
Q_LO_IDX  = int(ALPHA / 2 * N_QUANTILES)
Q_HI_IDX  = int((1 - ALPHA / 2) * N_QUANTILES) - 1
MED_IDX   = N_QUANTILES // 2

cal_scores = np.array([
    max(q[Q_LO_IDX] - y, y - q[Q_HI_IDX])
    for _, y, q in cal_results
])
beta = np.quantile(cal_scores, 1 - ALPHA)
print(f'Global beta = {beta:.4f}')

test_intervals_cqr = [
    (stem, y, q[Q_LO_IDX] - beta, q[Q_HI_IDX] + beta)
    for stem, y, q in test_results
]
cqr_df = pd.DataFrame([
    {'stem': s, 'y_true': y, 'lo': lo, 'hi': hi, 'covered': lo<=y<=hi}
    for s, y, lo, hi in test_intervals_cqr
])
print(f'CQR: PICP={cqr_df["covered"].mean():.4f}  MPIW={(cqr_df["hi"]-cqr_df["lo"]).mean():.4f}')

Global beta = 14.9148
CQR: PICP=0.8800  MPIW=36.5307


In [ ]:
# CELL 10: MONDRIAN CONFORMAL PREDICTION
# Stratifies calibration set by depression severity (same 4 bins as FUQ)
# but computes ONE threshold per bin — no demographic fairness adjustment.
# This isolates how much of FUQ's improvement comes from severity-stratification
# alone vs the added fairness-optimization layer.

print('=' * 60)
print('MONDRIAN CONFORMAL PREDICTION (severity-stratified only)')
print('=' * 60)

cal_df = pd.DataFrame([{
    'stem': stem, 'y_true': y,
    'y_lo': q[Q_LO_IDX], 'y_hi': q[Q_HI_IDX],
    'r': max(q[Q_LO_IDX]-y, y-q[Q_HI_IDX])
} for stem, y, q in cal_results])

cal_sorted = cal_df.sort_values('y_true').reset_index(drop=True)
N_cal, bin_size = len(cal_sorted), len(cal_sorted)//M_BINS

mondrian_bins  = []
mondrian_r_hat = {}
for m in range(M_BINS):
    s  = m*bin_size
    e  = (m+1)*bin_size if m < M_BINS-1 else N_cal
    bd = cal_sorted.iloc[s:e]
    # Mondrian: one conformal threshold per bin = (1-alpha) quantile of
    # nonconformity scores WITHIN that bin only
    r_m = np.quantile(bd['r'].values, 1-ALPHA) if len(bd) > 0 else beta
    mondrian_bins.append({'m': m, 'l': bd['y_true'].min(), 'u': bd['y_true'].max(), 'r': r_m, 'N': len(bd)})
    mondrian_r_hat[m] = r_m
    print(f'Bin {m+1}: [{bd["y_true"].min():.1f}, {bd["y_true"].max():.1f}]  N={len(bd)}  r_bin={r_m:.4f}')

def mondrian_interval(y_lo, y_hi, y_true_for_bin_lookup, bins):
    # Find which bin the prediction falls near using quantile prediction
    # (use predicted median to assign bin, since true label unknown at test time
    #  in a real deployment — but for evaluation we can also test oracle binning)
    for b in bins:
        if b['l'] <= y_true_for_bin_lookup <= b['u']:
            r = b['r']
            return y_lo - r, y_hi + r
    # fallback: closest bin by distance
    closest = min(bins, key=lambda b: min(abs(y_true_for_bin_lookup-b['l']), abs(y_true_for_bin_lookup-b['u'])))
    r = closest['r']
    return y_lo - r, y_hi + r

mondrian_rows = []
for stem, y_true, q in test_results:
    y_lo, y_hi = q[Q_LO_IDX], q[Q_HI_IDX]
    y_pred_median = q[MED_IDX]
    # Bin assignment uses PREDICTED median (realistic deployment scenario,
    # not oracle true label) to avoid information leakage
    lo, hi = mondrian_interval(y_lo, y_hi, y_pred_median, mondrian_bins)
    mondrian_rows.append({
        'stem': stem, 'y_true': y_true, 'y_pred': y_pred_median,
        'lo': lo, 'hi': hi, 'covered': lo<=y_true<=hi,
        'gender': gender_map.get(stem, '?')
    })

mondrian_df = pd.DataFrame(mondrian_rows)
print(f'\n=== Mondrian CP Results ===')
print(f'Overall PICP: {mondrian_df["covered"].mean():.4f}')
print(f'Overall MPIW: {(mondrian_df["hi"]-mondrian_df["lo"]).mean():.4f}')
for g, lbl in [('F','Female'), ('M','Male')]:
    grp = mondrian_df[mondrian_df['gender']==g]
    if len(grp) == 0: continue
    print(f'  {lbl:6s} N={len(grp):3d}  PICP={grp["covered"].mean():.4f}  MPIW={(grp["hi"]-grp["lo"]).mean():.4f}')
picp_f = mondrian_df[mondrian_df.gender=='F']['covered'].mean()
picp_m = mondrian_df[mondrian_df.gender=='M']['covered'].mean()
print(f'  PICP Gap: {abs(picp_f-picp_m):.4f}')

MONDRIAN CONFORMAL PREDICTION (severity-stratified only)
Bin 1: [0.0, 3.0]  N=12  r_bin=15.0548
Bin 2: [3.0, 12.0]  N=12  r_bin=8.3498
Bin 3: [12.0, 21.0]  N=12  r_bin=7.3336
Bin 4: [22.0, 43.0]  N=14  r_bin=16.3131

=== Mondrian CP Results ===
Overall PICP: 0.7800
Overall MPIW: 26.1729
  Female N= 15  PICP=0.7333  MPIW=27.4620
  Male   N= 35  PICP=0.8000  MPIW=25.6204
  PICP Gap: 0.0667


In [ ]:
# CELL 11: POST-HOC RECALIBRATION — Temperature Scaling
# Learns a single scalar T fit on the calibration set such that
# scaling the interval half-width by T achieves target coverage (0.90).
# q_scaled_lo = median - T*(median - y_lo)
# q_scaled_hi = median + T*(y_hi - median)

print('=' * 60)
print('POST-HOC RECALIBRATION: Temperature Scaling')
print('=' * 60)

def temp_scale_coverage(T, results):
    covered = []
    for stem, y, q in results:
        med   = q[MED_IDX]
        lo    = med - T*(med - q[Q_LO_IDX])
        hi    = med + T*(q[Q_HI_IDX] - med)
        covered.append(lo <= y <= hi)
    return np.mean(covered)

# Search for T on calibration set that hits target coverage
T_grid = np.arange(0.5, 3.01, 0.01)
cal_coverages = [temp_scale_coverage(T, cal_results) for T in T_grid]

# Pick smallest T that reaches >= target (most efficient / narrowest)
valid_T = [T for T, c in zip(T_grid, cal_coverages) if c >= TARGET]
T_star  = min(valid_T) if valid_T else T_grid[np.argmax(cal_coverages)]

cal_cov_at_T  = temp_scale_coverage(T_star, cal_results)
print(f'Optimal temperature T* = {T_star:.2f}')
print(f'Calibration coverage at T*: {cal_cov_at_T:.4f}  (target {TARGET:.2f})')

# Apply to test set
temp_rows = []
for stem, y, q in test_results:
    med = q[MED_IDX]
    lo  = med - T_star*(med - q[Q_LO_IDX])
    hi  = med + T_star*(q[Q_HI_IDX] - med)
    temp_rows.append({
        'stem': stem, 'y_true': y, 'y_pred': med,
        'lo': lo, 'hi': hi, 'covered': lo<=y<=hi,
        'gender': gender_map.get(stem, '?')
    })
temp_df = pd.DataFrame(temp_rows)

print(f'\n=== Temperature-Scaled Results (test set) ===')
print(f'Overall PICP: {temp_df["covered"].mean():.4f}  (target {TARGET:.2f})')
print(f'Overall MPIW: {(temp_df["hi"]-temp_df["lo"]).mean():.4f}')
print(f'Raw (uncalibrated) PICP was: {np.mean([q[Q_LO_IDX]<=y<=q[Q_HI_IDX] for _,y,q in test_results]):.4f}')

POST-HOC RECALIBRATION: Temperature Scaling
Optimal temperature T* = 2.89
Calibration coverage at T*: 0.6400  (target 0.90)

=== Temperature-Scaled Results (test set) ===
Overall PICP: 0.6000  (target 0.90)
Overall MPIW: 19.3663
Raw (uncalibrated) PICP was: 0.2400


In [ ]:
# CELL 12: POST-HOC RECALIBRATION — Isotonic Regression
# Fits a monotonic mapping from raw quantile levels to empirical
# coverage levels observed on the calibration set, then uses the
# inverse mapping to find which RAW quantile indices to use as
# bounds so that empirical test coverage hits target.

from sklearn.isotonic import IsotonicRegression

print('=' * 60)
print('POST-HOC RECALIBRATION: Isotonic Regression')
print('=' * 60)

# For each nominal quantile level, compute empirical coverage on cal set
# i.e. what fraction of cal samples have y_true <= predicted quantile q_i
nominal_levels = np.linspace(0.01, 0.99, N_QUANTILES)
empirical_cov  = []
for i, level in enumerate(nominal_levels):
    frac_below = np.mean([y <= q[i] for _, y, q in cal_results])
    empirical_cov.append(frac_below)
empirical_cov = np.array(empirical_cov)

# Fit isotonic regression: nominal level -> empirical coverage
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(nominal_levels, empirical_cov)

# Find recalibrated quantile indices for alpha/2 and 1-alpha/2 targets
# by inverting: which nominal level gives empirical coverage = alpha/2 ?
target_lo = ALPHA / 2
target_hi = 1 - ALPHA / 2

# Search nominal levels for the ones whose calibrated (isotonic) output
# is closest to target_lo / target_hi
iso_preds = iso.predict(nominal_levels)
idx_lo = np.argmin(np.abs(iso_preds - target_lo))
idx_hi = np.argmin(np.abs(iso_preds - target_hi))

print(f'Recalibrated lower quantile index: {idx_lo}  (nominal level={nominal_levels[idx_lo]:.3f}, isotonic coverage={iso_preds[idx_lo]:.3f})')
print(f'Recalibrated upper quantile index: {idx_hi}  (nominal level={nominal_levels[idx_hi]:.3f}, isotonic coverage={iso_preds[idx_hi]:.3f})')

iso_rows = []
for stem, y, q in test_results:
    lo = q[idx_lo]
    hi = q[idx_hi]
    if lo > hi:
        lo, hi = hi, lo
    iso_rows.append({
        'stem': stem, 'y_true': y, 'y_pred': q[MED_IDX],
        'lo': lo, 'hi': hi, 'covered': lo<=y<=hi,
        'gender': gender_map.get(stem, '?')
    })
iso_df = pd.DataFrame(iso_rows)

print(f'\n=== Isotonic-Recalibrated Results (test set) ===')
print(f'Overall PICP: {iso_df["covered"].mean():.4f}  (target {TARGET:.2f})')
print(f'Overall MPIW: {(iso_df["hi"]-iso_df["lo"]).mean():.4f}')

POST-HOC RECALIBRATION: Isotonic Regression
Recalibrated lower quantile index: 0  (nominal level=0.010, isotonic coverage=0.300)
Recalibrated upper quantile index: 98  (nominal level=0.990, isotonic coverage=0.740)

=== Isotonic-Recalibrated Results (test set) ===
Overall PICP: 0.4000  (target 0.90)
Overall MPIW: 10.5018


In [ ]:
# CELL 13: Full comparison table — all methods side by side
print('=' * 78)
print('FULL METHOD COMPARISON — AVEC 2014 test set')
print('=' * 78)
print(f'{"Method":30s} {"PICP":>8s} {"MPIW":>8s} {"PICP(F)":>9s} {"PICP(M)":>9s} {"Gap":>8s}')
print('-' * 78)

def row_stats(df, name):
    picp = df['covered'].mean()
    mpiw = (df['hi']-df['lo']).mean()
    pf = df[df.gender=='F']['covered'].mean() if 'gender' in df and (df.gender=='F').any() else float('nan')
    pm = df[df.gender=='M']['covered'].mean() if 'gender' in df and (df.gender=='M').any() else float('nan')
    gap = abs(pf-pm) if not (np.isnan(pf) or np.isnan(pm)) else float('nan')
    print(f'{name:30s} {picp:8.4f} {mpiw:8.4f} {pf:9.4f} {pm:9.4f} {gap:8.4f}')

cqr_df['gender'] = cqr_df['stem'].map(gender_map)
row_stats(cqr_df,      'CQR (global)')
row_stats(mondrian_df, 'Mondrian CP (severity only)')
row_stats(temp_df,     'Temperature Scaling')
row_stats(iso_df,      'Isotonic Regression')

print('\nNote: FUQ (gender-fair) results from prior notebook: PICP=0.88, MPIW=30.20, Gap=0.019')

FULL METHOD COMPARISON — AVEC 2014 test set
Method                             PICP     MPIW   PICP(F)   PICP(M)      Gap
------------------------------------------------------------------------------
CQR (global)                     0.8800  36.5307    0.8667    0.8857   0.0190
Mondrian CP (severity only)      0.7800  26.1729    0.7333    0.8000   0.0667
Temperature Scaling              0.6000  19.3663    0.6000    0.6000   0.0000
Isotonic Regression              0.4000  10.5018    0.4667    0.3714   0.0952

Note: FUQ (gender-fair) results from prior notebook: PICP=0.88, MPIW=30.20, Gap=0.019


In [ ]:
# CELL 14: Save all results
mondrian_df.to_csv(os.path.join(SAVE_DIR, 'mondrian_results.csv'), index=False)
temp_df.to_csv(os.path.join(SAVE_DIR, 'temperature_scaling_results.csv'), index=False)
iso_df.to_csv(os.path.join(SAVE_DIR, 'isotonic_results.csv'), index=False)
print(f'All results saved to: {SAVE_DIR}')

print('\n===== Final Summary =====')
print(f'CQR                  PICP={cqr_df["covered"].mean():.4f}  MPIW={(cqr_df["hi"]-cqr_df["lo"]).mean():.4f}')
print(f'Mondrian CP          PICP={mondrian_df["covered"].mean():.4f}  MPIW={(mondrian_df["hi"]-mondrian_df["lo"]).mean():.4f}')
print(f'Temperature Scaling  PICP={temp_df["covered"].mean():.4f}  MPIW={(temp_df["hi"]-temp_df["lo"]).mean():.4f}  (T*={T_star:.2f})')
print(f'Isotonic Regression  PICP={iso_df["covered"].mean():.4f}  MPIW={(iso_df["hi"]-iso_df["lo"]).mean():.4f}')

All results saved to: /content/drive/MyDrive/avec2014_mondrian_recal

===== Final Summary =====
CQR                  PICP=0.8800  MPIW=36.5307
Mondrian CP          PICP=0.7800  MPIW=26.1729
Temperature Scaling  PICP=0.6000  MPIW=19.3663  (T*=2.89)
Isotonic Regression  PICP=0.4000  MPIW=10.5018
